## 🏃‍♂️ Sprint 1: Data Ingestion & Regional Expansion

**Current Status:** Completed.

**Sprint Goal:** Successfully ingest the baseline hardware pricing dataset and transform the broad macro-regions into explicit country-level rows with calculated local currency and tax metrics.

### **US-1.1: Raw Data Ingestion**

* **Status:** Completed
* **Role/Persona:** Data Engineer
* **Problem Statement:** As a data engineer, I need to configure the Kaggle API and download the raw hardware crisis pricing dataset into our local environment so that we have an immutable data source to build our pipeline upon.
* **Acceptance Criteria:**
* Securely configure the modern Kaggle `access_token` setup in the local user directory (`~/.kaggle/`).
* Successfully download and unzip `moaz1911/ultimate-memory-crisis-hardware-market-dynamics` to `data/raw/` using a command-line script.
* Verify data integrity by confirming a clean data import with zero null values across all 28 baseline columns.



---

### **US-1.2: Regional Data Explosion & Localization (Architectural Pivot: DataFrame Merge)**

* **Status:** 🟢 Completed
* **Role/Persona:** Data Engineer / Analytics Engineer
* **Problem Statement:** As a data engineer, I need to convert the static regional dictionary into a reference DataFrame and use a relational merge to expand macro-regions into specific countries, allowing us to calculate localized target variables without slow row-by-row Python loops.
* **Acceptance Criteria:**
* Structure the original `regional_market_map` nested dictionary into a flat reference Pandas DataFrame containing the columns: `region`, `country`, `country_code`, `tax_rate`, `currency_rate`, and `currency_code`.
* Perform a vectorized relational join (`pd.merge`) between the main hardware dataset and the reference DataFrame using `region` as the join key to automatically handle row explosion.
* Compute a new vectorized target variable column named `final_local_price` using the formula: $\text{final\_local\_price} = (\text{price\_usd} \times \text{tax\_rate}) \times \text{currency\_rate}$.


* **Mock Input Data Slice:**
```json
{
  "timestamp": "2023-01-01",
  "model_name": "DDR5-6000",
  "price_usd": 100.00,
  "region": "North America"
}

```


* **Mock Output Data Slice:**
```json
[
  {
    "timestamp": "2023-01-01",
    "model_name": "DDR5-6000",
    "price_usd": 100.00,
    "region": "North America",
    "country": "United States",
    "country_code": "US",
    "currency_code": "USD",
    "final_local_price": 100.00
  },
  {
    "timestamp": "2023-01-01",
    "model_name": "DDR5-6000",
    "price_usd": 100.00,
    "region": "North America",
    "country": "Canada",
    "country_code": "CA",
    "currency_code": "CAD",
    "final_local_price": 153.44
  }
]

```
### **US-1.3: Data Quality Gatekeeper**
* **Status:** 🟢 Completed
* **Role/Persona:** Data Engineer / Quality Assurance
* **Problem Statement:** As a data engineer, I need to implement automated validation checks (completeness, consistency, and sanity) on transformed hardware data before exporting, so that corrupted, missing, or non-positive pricing records are caught and prevented from propagating downstream.
* **Acceptance Criteria:**
- Implement `validate_completeness` to verify no unexpected null values exist across critical columns.
- Implement `validate_consistency` to verify record counts and row expansion logic between raw and merged datasets.
- Implement `validate_sanity` to check for non-positive or logically invalid pricing values (`price_usd <= 0` or `final_local_price <= 0`).
- Combine validation functions into `run_pipeline_validations` acting as a fail-fast barrier prior to file export.

### **US-1.4: Production Modular Refactoring & Execution Entry Point**
* **Status:**  🟢 Completed
* **Role/Persona:** Data Engineer
* **Problem Statement:** As a data engineer, I need to refactor notebook-bound ETL and validation code into a modular Python package layout (`src/app/`) with a root-relative entry point (`src/main.py`), so that the pipeline can run reliably via `python -m src.main`.
* **Acceptance Criteria:**
- Separate validation logic into `src/app/validation.py` and pipeline execution steps into `src/app/pipeline.py`.
- Configure `src/main.py` with an `if __name__ == "__main__":` entry guard.
- Export clean, validated output to `data/processed/hardware_prices_processed.csv` with `index=False`.

---

## 🧊 Product Backlog (The Icebox / Future Enhancements)

### **ICE-1.1: Dynamic Reference Data Ingestion**

* **Status:** ❄️ Backlog / Deferred
* **Role/Persona:** Data Pipeline Engineer
* **Problem Statement:** As a data engineer, I want to automate the ingestion of global tax and currency rates from external APIs rather than hardcoding them, so that the pricing model stays accurate to real-world economic shifts without manual code intervention.
* **Acceptance Criteria:**
* Replace the static reference data steps with an external fetch script.
* Query live currency rates via an open exchange API (e.g., Frankfurter API) during the pipeline execution run.
* Automate tax data collection via a dedicated compliance API or scheduled web scrape of an aggregator (e.g., PwC/VATcomply).
* Build pipeline fault-tolerance by implementing a graceful fallback to a cached local reference CSV if the external API endpoints fail.